In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LongformerTokenizer, LongformerForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- Longformer Dataset ---
class LongformerTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        padding_strategy = "longest" if self.dynamic_padding else "max_length"
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)
        }
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            if self.global_attention_target == 'cls':
                global_attention_mask[0] = 1
            output['global_attention_mask'] = global_attention_mask
        return output

def get_longformer_predictions(model, data_loader, device):
    model.eval()
    predictions, all_probs = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            # labels = batch['labels'].to(device)  # not used for prediction
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            probabilities = torch.sigmoid(outputs.logits)
            predicted = (probabilities >= 0.5).float()
            predictions.extend(predicted.view(-1).cpu().numpy())
            all_probs.extend(probabilities.view(-1).cpu().numpy())
    return np.array(predictions), np.array(all_probs)

# --- Model configs: ONLY Longformer ---
longformer_model_configs = [
    {
        "name": "Longformer_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/mimic_no_preprocess_no_glob_binary",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_mimic_v010725",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_phenotype_to_mimic_v030325",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    }
]

In [4]:
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label'].tolist()

In [5]:
BATCH_SIZE = 48  # Longformer usually only fits 1 per GPU/CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Store model results
all_preds_dict = {}
all_probs_dict = {}

for cfg in longformer_model_configs:
    print(f"Evaluating model: {cfg['name']}")
    tokenizer = cfg['tokenizer_cls'].from_pretrained(cfg['model_dir'])
    model = cfg['model_cls'].from_pretrained(cfg['model_dir'])
    model.to(device)
    dataset = LongformerTextDataset(
        test_texts, test_labels, tokenizer,
        max_length=cfg['max_length'],
        use_global_attention=True,
        global_attention_target='cls',
        dynamic_padding=False
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    preds, probs = get_longformer_predictions(model, loader, device)
    all_preds_dict[cfg['name']] = preds
    all_probs_dict[cfg['name']] = probs

# Assemble wide-format results
wide_results = pd.DataFrame({
    'true_label': test_labels
})
for model_name in all_preds_dict:
    wide_results[f'{model_name}_pred'] = all_preds_dict[model_name]
    wide_results[f'{model_name}_prob'] = all_probs_dict[model_name]

Evaluating model: Longformer_MIMIC


Predicting: 100%|██████████| 18/18 [01:35<00:00,  5.28s/it]


Evaluating model: Longformer_Berkeley_MIMIC


Predicting: 100%|██████████| 18/18 [01:33<00:00,  5.22s/it]


Evaluating model: Longformer_Berkeley_Phenotype_MIMIC


Predicting: 100%|██████████| 18/18 [01:34<00:00,  5.23s/it]


In [6]:
wide_results

,true_label,Longformer_MIMIC_pred,Longformer_MIMIC_prob,Longformer_Berkeley_MIMIC_pred,Longformer_Berkeley_MIMIC_prob,Longformer_Berkeley_Phenotype_MIMIC_pred,Longformer_Berkeley_Phenotype_MIMIC_prob
0,1,1.0,0.982605,1.0,0.986531,1.0,0.995937
1,1,1.0,0.980281,1.0,0.978088,1.0,0.997577
2,0,0.0,0.145869,0.0,0.005442,0.0,0.003175
3,0,0.0,0.117038,0.0,0.004038,0.0,0.002465
4,1,1.0,0.985094,1.0,0.995011,1.0,0.998071
...,...,...,...,...,...,...,...
821,1,1.0,0.944428,1.0,0.965867,1.0,0.997234
822,1,1.0,0.979169,1.0,0.994320,1.0,0.997935
823,1,1.0,0.985285,1.0,0.993122,1.0,0.997935
824,0,0.0,0.029080,0.0,0.005554,0.0,0.001854


In [7]:
# Remove probability columns (those ending with '_prob')
cols_to_keep = ['true_label'] + [col for col in wide_results.columns if col.endswith('_pred')]
wide_preds = wide_results[cols_to_keep]
wide_preds

,true_label,Longformer_MIMIC_pred,Longformer_Berkeley_MIMIC_pred,Longformer_Berkeley_Phenotype_MIMIC_pred
0,1,1.0,1.0,1.0
1,1,1.0,1.0,1.0
2,0,0.0,0.0,0.0
3,0,0.0,0.0,0.0
4,1,1.0,1.0,1.0
...,...,...,...,...
821,1,1.0,1.0,1.0
822,1,1.0,1.0,1.0
823,1,1.0,1.0,1.0
824,0,0.0,0.0,0.0


In [8]:
wide_preds = wide_preds.astype(int)
wide_preds

,true_label,Longformer_MIMIC_pred,Longformer_Berkeley_MIMIC_pred,Longformer_Berkeley_Phenotype_MIMIC_pred
0,1,1,1,1
1,1,1,1,1
2,0,0,0,0
3,0,0,0,0
4,1,1,1,1
...,...,...,...,...
821,1,1,1,1
822,1,1,1,1
823,1,1,1,1
824,0,0,0,0


In [9]:
model_pred_cols = [col for col in wide_preds.columns if col.endswith('_pred')]

# Calculate accuracy for each model
accuracies = {}
for col in model_pred_cols:
    acc = (wide_preds[col] == wide_preds['true_label']).mean()
    # Remove the '_pred' suffix for cleaner model name
    accuracies[col.replace('_pred', '')] = acc

# Display as DataFrame
acc_df = pd.DataFrame(list(accuracies.items()), columns=['model', 'accuracy'])
print(acc_df)

                                 model  accuracy
0                     Longformer_MIMIC  0.859564
1            Longformer_Berkeley_MIMIC  0.877724
2  Longformer_Berkeley_Phenotype_MIMIC  0.897094


In [10]:
wide_preds.to_csv("/content/drive/My Drive/EHR_PROJ/Results/all_longformer_wide_preds.csv", index=False)